# Laboratorium 3: Od tokenów do reprezentacji wektorowych

W tym laboratorium porównamy rzadkie i gęste reprezentacje wektorowe oraz sprawdzimy, jak zachowują się w zadaniach opartych na podobieństwie tekstów.

Porównujemy:

- TF-IDF,
- `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`,
- `intfloat/multilingual-e5-small`.

Główne pytanie laboratorium:

**Czy bliskość dwóch tekstów w przestrzeni wektorowej odpowiada podobieństwu, którego potrzebujemy w konkretnym zadaniu NLP?**


## Po tych zajęciach student potrafi

- wyjaśnić różnicę między rzadką i gęstą reprezentacją wektorową;
- obliczyć i zinterpretować podobieństwo cosinusowe;
- wygenerować reprezentacje tekstu za pomocą modelu embeddingowego;
- zaimplementować wyszukiwanie najbliższych sąsiadów;
- ocenić jakość lokalnego sąsiedztwa;
- zbudować prosty benchmark wyszukiwania;
- zastosować PCA do analizy przestrzeni reprezentacji;
- rozpoznać przypadki, w których wysokie podobieństwo wektorowe nie odpowiada wymaganiom zadania;
- porównać jakość reprezentacji z jej kosztem obliczeniowym;
- uzasadnić wybór reprezentacji dla konkretnego zastosowania.


## Dane

Pracujemy na zbiorze PolEmo 2.0 wykorzystywanym w poprzednich laboratoriach.

Jeżeli istnieje plik:

```text
artifacts/polemo2_clean.parquet
```

należy użyć tej wersji danych. Jeśli pliku nie ma, proszę wrócić do Lab 1 i go przygotować.

W tym laboratorium nie używamy zbioru testowego do wyboru reprezentacji ani do strojenia eksperymentów.


## Ustawienia wstępne

Wymagane biblioteki:

```bash
pip install pandas numpy matplotlib scikit-learn pyarrow sentence-transformers
```

Kod startowy obejmuje wyłącznie importy, stałe i wczytanie danych. Implementacja eksperymentów należy do Państwa.


In [29]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import math
import torch
from scipy import sparse

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

from sentence_transformers import SentenceTransformer

RANDOM_STATE = 42
SAMPLE_SIZE = 1000

ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

MODEL_A = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODEL_B = "intfloat/multilingual-e5-small"

pd.set_option("display.max_colwidth", 180)


In [2]:
DATA_PATH = ARTIFACTS / "polemo2_clean.parquet"
assert DATA_PATH.exists(), "Brak artifacts/polemo2_clean.parquet. Proszę wykonać Lab 1."

data = pd.read_parquet(DATA_PATH).dropna(subset=["text", "target"]).reset_index(drop=True)
assert set(data["target"]) == {"plus", "minus", "zero", "amb"}

train_df = data[data["split"] == "train"].reset_index(drop=True)  # tylko train, bez validation i test
print(train_df.shape)
display(train_df.head())

(5783, 7)


,text,target,split,text_norm,dup_within_split,dup_cross_split,near_dup_cross_split
0,"Super lekarz i człowiek przez duże C . Bardzo duże doświadczenie i trafne diagnozy . Wielka cierpliwość do ludzi starszych . Od lat opiekuje się moją Mamą staruszką , i twierdz...",plus,train,"super lekarz i człowiek przez duże c . bardzo duże doświadczenie i trafne diagnozy . wielka cierpliwość do ludzi starszych . od lat opiekuje się moją mamą staruszką , i twierdz...",False,False,False
1,Bardzo olewcze podejscie do pacjenta . Przyprowadzajac dziecko z ostra wysypka na calym ciele trwajaca od 2 tygodni Pani doktor stwierdzila ze nie widzi wskazan wystawienia dzi...,minus,train,bardzo olewcze podejscie do pacjenta . przyprowadzajac dziecko z ostra wysypka na calym ciele trwajaca od 2 tygodni pani doktor stwierdzila ze nie widzi wskazan wystawienia dzi...,False,False,False
2,"Lekarz zalecił mi kurację alternatywną do dotychczasowej , więc jeszcze nie daję najwyższej oceny ( zobaczymy na ile okaże się skuteczna ) . Do Pana doktora nie mam zastrzeżeń ...",amb,train,"lekarz zalecił mi kurację alternatywną do dotychczasowej , więc jeszcze nie daję najwyższej oceny ( zobaczymy na ile okaże się skuteczna ) . do pana doktora nie mam zastrzeżeń ...",False,False,False
3,""" Konsumenci oczywiście kierują się ceną . Te leki są tańsze , ale dzisiaj nie możemy już powiedzieć , że znacznie tańsze . Jest to mniej krępujące , nie trzeba wychodzić z dom...",zero,train,""" konsumenci oczywiście kierują się ceną . te leki są tańsze , ale dzisiaj nie możemy już powiedzieć , że znacznie tańsze . jest to mniej krępujące , nie trzeba wychodzić z dom...",False,False,False
4,"Pani Doktor Iwona jest profesjonalistką w każdym calu : ) Idąc do stomatologa zawsze czuję przeraźliwy strach . Pani Doktor Iwona jest wyrozumiała i delikatna , bardzo miła , c...",plus,train,"pani doktor iwona jest profesjonalistką w każdym calu : ) idąc do stomatologa zawsze czuję przeraźliwy strach . pani doktor iwona jest wyrozumiała i delikatna , bardzo miła , c...",False,False,False


# Zadanie 1: TF-IDF i podobieństwo cosinusowe

Proszę rozpocząć od tekstów:

```text
A = "hotel czysty wygodny"
B = "hotel bardzo czysty"
C = "samochód szybki sportowy"
```

Proszę zaimplementować:

1. reprezentację TF-IDF;
2. tabelę wartości TF-IDF;
3. macierz podobieństwa cosinusowego;
4. wskazanie najbardziej podobnej pary.

Następnie proszę powtórzyć eksperyment dla:

```text
A = "Hotel był bardzo dobry."
B = "Obiekt noclegowy zrobił na mnie świetne wrażenie."
C = "Hotel był bardzo zły."
```

Przed uruchomieniem kodu proszę zapisać, która para powinna być najbardziej podobna według TF-IDF, a która według człowieka.


In [3]:
A = "hotel czysty wygodny"
B = "hotel bardzo czysty"
C = "samochód szybki sportowy"

A2 = "Hotel był bardzo dobry."
B2 = "Obiekt noclegowy zrobił na mnie świetne wrażenie."
C2 = "Hotel był bardzo zły."

In [4]:
class MyTfidf:
    def __init__(self, normalize: bool = True):
        self.normalize = normalize
        self.vocab_: dict[str, int] | None = None
        self.idf_: np.ndarray | None = None


    def fit(self, texts: list[str]) -> "MyTfidf":
        docs_tokens = [self._tokenize(t) for t in texts]
        self.vocab_ = self._build_vocab(docs_tokens)
        self.idf_ = self._build_idf(docs_tokens)
        return self

    def transform(self, texts: list[str]) -> np.ndarray:
        docs_tokens = [self._tokenize(t) for t in texts]
        X = self._build_tf(docs_tokens) * self.idf_
        return self._l2_normalize(X) if self.normalize else X

    def fit_transform(self, texts: list[str]) -> np.ndarray:
        return self.fit(texts).transform(texts)

    def get_feature_names_out(self) -> list[str]:
        return sorted(self.vocab_, key=self.vocab_.get)


    @staticmethod
    def _tokenize(text: str) -> list[str]:
        return re.findall(r"\w+", text.lower())

    @staticmethod
    def _build_vocab(docs_tokens: list[list[str]]) -> dict[str, int]:
        vocab = {}
        for tokens in docs_tokens:
            for word in tokens:
                vocab.setdefault(word, len(vocab))
        return vocab

    @staticmethod
    def _l2_normalize(matrix: np.ndarray) -> np.ndarray:
        norms = np.linalg.norm(matrix, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        return matrix / norms

    def _build_tf(self, docs_tokens: list[list[str]]) -> np.ndarray:
        tf = np.zeros((len(docs_tokens), len(self.vocab_)))
        for i, tokens in enumerate(docs_tokens):
            for word in tokens:
                if word in self.vocab_:
                    tf[i, self.vocab_[word]] += 1
        return tf

    def _build_idf(self, docs_tokens: list[list[str]]) -> np.ndarray:
        n_docs = len(docs_tokens)
        df = np.zeros(len(self.vocab_))
        for tokens in docs_tokens:
            for word in set(tokens):
                df[self.vocab_[word]] += 1
        return np.log((1 + n_docs) / (1 + df)) + 1

In [5]:
def cosine_sim(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    matrix = matrix / norms
    return matrix @ matrix.T


def ranked_pairs(sim: np.ndarray, labels: list[str]) -> list[tuple[str, float]]:
    i, j = np.triu_indices(len(sim), k=1)
    order = np.argsort(sim[i, j])[::-1]
    return [(f"{labels[i[o]]}-{labels[j[o]]}", float(sim[i[o], j[o]])) for o in order]

In [6]:
def run_experiment(vectorizer, texts: list[str], labels: list[str]) -> None:
    X = vectorizer.fit_transform(texts)
    X.toarray() if hasattr(X, "toarray") else X

    features = vectorizer.get_feature_names_out()

    sim = cosine_sim(X)
    pair, score = ranked_pairs(sim, labels)[0]

    display(pd.DataFrame(X, index=labels, columns=features).round(3))
    display(pd.DataFrame(sim, index=labels, columns=labels).round(3))
    print(f"Najbardziej podobna para: {pair} (cos = {score:.3f})")

> **Eksperyment 1**
>
> Teksty:
> * A = "hotel czysty wygodny"
> * B = "hotel bardzo czysty"
> * C = "samochód szybki sportowy"
>
> Według człowieka najbardziej pdoobne powinny być teskty A i B, według TF-IDF również.

In [7]:
run_experiment(MyTfidf(), [A, B, C], ["A", "B", "C"])

,hotel,czysty,wygodny,bardzo,samochód,szybki,sportowy
A,0.518,0.518,0.681,0.000,0.000,0.000,0.000
B,0.518,0.518,0.000,0.681,0.000,0.000,0.000
C,0.000,0.000,0.000,0.000,0.577,0.577,0.577


,A,B,C
A,1.000,0.536,0.0
B,0.536,1.000,0.0
C,0.000,0.000,1.0


Najbardziej podobna para: A-B (cos = 0.536)


> Wynik taki jak oczekiwany.

> **Eksperyment 2**
>
> Teksty:
> * A = "Hotel był bardzo dobry."
> * B = "Obiekt noclegowy zrobił na mnie świetne wrażenie."
> * C = "Hotel był bardzo zły."
>
> Według człowieka najbardziej pdoobne powinny być teskty A i B, natomiast według TF-IDF będą to zapewne teskty A i C.

In [8]:
run_experiment(MyTfidf(), [A2, B2, C2], ["A", "B", "C"])

,hotel,był,bardzo,dobry,obiekt,noclegowy,zrobił,na,mnie,świetne,wrażenie,zły
A,0.46,0.46,0.46,0.605,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
B,0.00,0.00,0.00,0.000,0.378,0.378,0.378,0.378,0.378,0.378,0.378,0.000
C,0.46,0.46,0.46,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.605


,A,B,C
A,1.000,0.0,0.634
B,0.000,1.0,0.000
C,0.634,0.0,1.000


Najbardziej podobna para: A-C (cos = 0.634)


> No i faktycznie według TF-IDF najbardziej podobne okazazły się teskty A i C, mimo że przekazują one całkiem inna opinie, są jednak zbudowane z prawie identycznych słów.

# Zadanie 2: Gęste reprezentacje wektorowe

Proszę wczytać modele `MODEL_A` i `MODEL_B` oraz wygenerować reprezentacje trzech zdań z drugiej części zadania 1.

Dla obu modeli proszę:

1. sprawdzić wymiar wektora;
2. sprawdzić typ danych;
3. wygenerować reprezentacje z normalizacją;
4. obliczyć macierz podobieństwa cosinusowego;
5. porównać kolejność podobieństw z wynikiem TF-IDF.

Model E5 wymaga prefiksów. Reguła (z karty modelu): zadania symetryczne (podobieństwo dwóch tekstów, sąsiedzi, PCA) → `query: ...` po obu stronach; wyszukiwanie (Zadanie 6) → `query: ...` dla zapytania i `passage: ...` dla dokumentów.

W tym zadaniu porównujemy zdania między sobą, więc proszę użyć `query: ...` dla wszystkich trzech zdań.

Dla obu modeli proszę dodatkowo sprawdzić `model.max_seq_length` oraz, dla tekstów z próbki z Zadania 3, jaki odsetek jest dłuższy niż okno modelu (po tokenizacji). Jak to może wpływać na kolejne zadania?

Wynik proszę zebrać w krótkiej tabeli porównawczej.


In [9]:
models = {
    "MiniLM": SentenceTransformer(MODEL_A),
    "E5": SentenceTransformer(MODEL_B),
    "TF-IDF": MyTfidf()
}

inputs = {
    "MiniLM": [A2, B2, C2],
    "E5": [f"query: {text}" for text in [A2, B2, C2]],
    "TF-IDF": [A2, B2, C2]
}

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4282.10it/s]


In [32]:
sample_df, _ = train_test_split(train_df, train_size=SAMPLE_SIZE, stratify=train_df["target"], random_state=RANDOM_STATE)
sample_df = sample_df.reset_index(drop=True)

texts = sample_df["text"].tolist()

sample_inputs = {
    "TF-IDF": texts,
    "MiniLM": texts,
    "E5": [f"query: {t}" for t in texts],
}

> Próbki do zadania 3, potrzebne tutaj do tabeli porównawczej.

In [38]:
results = []
for model_name, model in models.items():
    input_ = inputs[model_name]
    if model_name == "TF-IDF":
        output = model.fit_transform(input_)
        max_seq_length = "-"
        texts_longer_than_window = "-"
    else:
        output = model.encode(input_, normalize_embeddings=True)
        max_seq_length = f"{model.max_seq_length}"
        lens = [len(model.tokenizer(i, verbose=False)["input_ids"]) for i in sample_inputs[model_name]]
        texts_longer_than_window = f"{(np.array(lens) > model.max_seq_length).mean() * 100:.1f}"
        
    sim = cosine_sim(output)
    pairs = ranked_pairs(sim, ["A", "B", "C"])                          

    results.append({
        "Model": model_name,
        "Data Type": output.dtype,
        "Data Shape": output.shape[1],
        "Ranking": " > ".join(p for p, _ in pairs),
        "Similarity": [round(s, 3) for _, s in pairs],
        "Max Sequence Length": max_seq_length,
        "% of texts > window": texts_longer_than_window
    })

pd.DataFrame(results)

,Model,Data Type,Data Shape,Ranking,Similarity,Max Sequence Length,% of texts > window
0,MiniLM,float32,384,A-B > A-C > B-C,"[0.749, 0.378, 0.241]",128,76.7
1,E5,float32,384,A-B > A-C > B-C,"[0.927, 0.909, 0.869]",512,3.7
2,TF-IDF,float64,12,A-C > B-C > A-B,"[0.634, 0.0, 0.0]",-,-


> Oba modele embeddingowe stawiają A-B na pierwszym miejscu, więc rozpoznają podobieństwo („obiekt noclegowy” ≈ „hotel”), której TF-IDF w ogóle nie widzi.
>
> E5 daje wszystkie wartości w wąskim przedziale 0.87–0.93.
>
> Kolejność remisów w TF-IDF jest przypadkowa. Fragment B-C > A-B nic nie znaczy przy wartościach [0.0, 0.0].
>
> MiniLM ma okno 128 tokenów i obcina aż 76.7% recenzji z próbki, więc dla większości tekstów widzi tylko ich początek. E5 (512 tokenów) obcina tylko 3.7%.

# Zadanie 3: Próbka danych i koszt reprezentacji

Proszę przygotować deterministyczną próbkę `SAMPLE_SIZE` dokumentów ze zbioru treningowego. Próbka powinna w przybliżeniu zachowywać proporcje klas.

Następnie proszę utworzyć dla tej samej próbki:

- TF-IDF z `ngram_range=(1, 2)`, `min_df=2`, `max_features=50_000`;
- reprezentacje MiniLM;
- reprezentacje E5 z prefiksem `query: ` (wersja symetryczna, używana w Zadaniach 4, 5, 7 i 8; dla Zadania 6 dokumenty trzeba będzie zakodować ponownie z prefiksem `passage: `, ten czas nie wchodzi do tabeli).

Przy pomiarze czasu proszę wyłączyć ładowanie modelu, zrobić jedną rozgrzewkę (pojedyncze uruchomienie) na małej partii i zapisać urządzenie (CPU/GPU).

Dla każdej reprezentacji proszę zmierzyć:

- czas utworzenia;
- średni czas na dokument;
- liczbę wymiarów;
- typ reprezentacji: rzadka lub gęsta;
- rozmiar reprezentacji w pamięci.

Dla macierzy rzadkiej należy uwzględnić rzeczywisty koszt struktury sparse, a nie rozmiar macierzy po konwersji do postaci gęstej.

Wynik:

| reprezentacja | czas [s] | ms/dokument | liczba wymiarów | rozmiar [MB] | typ |
|---|---:|---:|---:|---:|---|
| TF-IDF | | | | | |
| MiniLM | | | | | |
| E5 | | | | | |


In [39]:
sample_models = {
    "TF-IDF": TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50_000),
    "MiniLM": models["MiniLM"],
    "E5": models["E5"],
}

pd.DataFrame({
    "train": train_df["target"].value_counts(normalize=True),
    "sample": sample_df["target"].value_counts(normalize=True),
}).round(3)

,train,sample
target,,
minus,0.379,0.379
plus,0.271,0.271
amb,0.182,0.182
zero,0.168,0.168


In [40]:
torch.cuda.is_available(), torch.cuda.get_device_name(0)

(True, 'NVIDIA GeForce RTX 5070 Ti')

In [41]:
def memory_mb(X) -> float:
    if sparse.issparse(X):
        nbytes = X.data.nbytes + X.indices.nbytes + X.indptr.nbytes
    else:
        nbytes = X.nbytes
    return nbytes / 1024**2


def device_of(model) -> str:
    return "GPU" if getattr(model, "device", None) is not None and model.device.type == "cuda" else "CPU"

In [42]:
SMALL_PORTION = int(SAMPLE_SIZE * 0.01)

encoders = {
    "TF-IDF": lambda t: sample_models["TF-IDF"].fit_transform(t),
    "MiniLM": lambda t: sample_models["MiniLM"].encode(t, normalize_embeddings=True),
    "E5": lambda t: sample_models["E5"].encode(t, normalize_embeddings=True),
}

representations = {}
sample_results = []

for name, encode in encoders.items():
    inputs_ = sample_inputs[name]

    encode(inputs_[:SMALL_PORTION])

    start = time.perf_counter()
    output = encode(inputs_)
    end = time.perf_counter() - start

    representations[name] = output
    sample_results.append({
        "reprezentacja": name,
        "czas [s]": round(end, 3),
        "ms/dokument": round(end / len(inputs_) * 1000, 3),
        "liczba wymiarów": output.shape[1],
        "rozmiar [MB]": round(memory_mb(output), 3),
        "typ": "rzadka" if sparse.issparse(output) else "gęsta",
        "urządzenie": device_of(sample_models[name]),
    })

pd.DataFrame(sample_results)

,reprezentacja,czas [s],ms/dokument,liczba wymiarów,rozmiar [MB],typ,urządzenie
0,TF-IDF,0.192,0.192,15649,1.130,rzadka,CPU
1,MiniLM,0.520,0.520,384,1.465,gęsta,GPU
2,E5,1.094,1.094,384,1.465,gęsta,GPU


> TF-IDF znacznie więcej wymiarów niż embeddingi, a zajmuje mniej pamięci, bo macierz rzadka przechowuje tylko niezerowe wartości, więc realnie jest to pewnie tylko mały procent całej macierzy.
>
> TF-IDF na CPU jest 2.5–5× szybszy niż embeddingi liczone na GPU.
>
> E5 jest ~2× wolniejszy od MiniLM przy tym samym wymiarze, prawdopodobnie przez dłuższe okno (512 vs 128 tokenów), czyli więcej przetwarzanych tokenów.

# Zadanie 4: Najbliżsi sąsiedzi

Proszę zaimplementować funkcję:

```python
get_neighbours(embeddings, row_id, k=5)
```

która zwraca identyfikatory najbliższych dokumentów oraz wartość podobieństwa. Funkcja nie może zwracać samego dokumentu jako sąsiada. Proszę pobrać k+1 sąsiadów i usunąć z wyniku indeks równy row_id, nie zakładając, że jest on pierwszy.

Proszę wybrać trzy dokumenty z różnych klas i dla każdego porównać pięciu najbliższych sąsiadów (teksty proszę skrócić do pierwszych 200 znaków) dla:
- TF-IDF,
- MiniLM,
- E5.

Dla każdego dokumentu proszę krótko zaznaczyć:

- czy sąsiedzi są podobni tematycznie;
- czy mają tę samą etykietę sentymentu;
- czy podobieństwo wynika głównie ze wspólnego słownictwa.

In [43]:
def get_neighbours(embeddings, row_id: int, k: int = 5) -> tuple[np.ndarray, np.ndarray]:
    nn = NearestNeighbors(metric="cosine").fit(embeddings)
    dists, inds = nn.kneighbors(embeddings[row_id:row_id + 1], n_neighbors=k + 1)

    sims, inds = 1 - dists[0], inds[0]
    mask = inds != row_id
    return inds[mask][:k], sims[mask][:k]

# Zadanie 5: Czystość sąsiedztwa

Dla dokumentu $x$:

$$
\text{purity@}k(x) = \frac{\text{liczba sąsiadów z tą samą etykietą}}{k}
$$

proszę policzyć średnią tej wartości po całej próbce (bez samego dokumentu jako sąsiada) dla k = 1, 5, 10 dla TF-IDF, MiniLM i E5. Dodatkowo proszę podać:

- punkt odniesienia: oczekiwaną purity dla losowych sąsiadów, czyli sumę kwadratów udziałów klas (`(train_df["target"].value_counts(normalize=True) ** 2).sum()`; przy rozkładzie z Lab 1 ok. 0,28);
- `purity@5` osobno dla każdej klasy.

Proszę policzyć:

- `purity@1`,
- `purity@5`,
- `purity@10`

dla TF-IDF, MiniLM i E5.

Wynik:

| reprezentacja | purity@1 | purity@5 | purity@10 |
|---|---:|---:|---:|
| TF-IDF | | | |
| MiniLM | | | |
| E5 | | | |


In [15]:
# Proszę zaimplementować obliczanie purity@k.

# Zadanie 6: Miniaturowy benchmark wyszukiwania

Proszę przygotować osiem zapytań: po dwa dla każdej klasy `plus`, `minus`, `zero`, `amb`.

Dla każdego zapytania należy zdefiniować oczekiwaną klasę dokumentów. Dla klas `zero` i `amb` proszę wcześniej zapisać, co w tym zbiorze znaczy "dokument relewantny" (pomoże przejrzenie kilkunastu losowych dokumentów tych klas z próbki).

Dla E5 dokumenty z próbki proszę zakodować ponownie z prefiksem `passage: ` (zapytania z `query: `).

Wyniki proszę podać również per zapytanie. Osiem zapytań to bardzo mała próba, więc różnic mniejszych niż ok. 0,1 między reprezentacjami nie należy interpretować jako przewagi.

Następnie proszę zaimplementować wyszukiwanie dla:

- TF-IDF;
- MiniLM;
- E5.

Dla E5 zapytanie powinno być kodowane z prefiksem:

```text
query: ...
```

a dokumenty z prefiksem:

```text
passage: ...
```

Proszę zaimplementować i policzyć:

- `Precision@5`;
- `Precision@10`.

Wynik:

| reprezentacja | Precision@5 | Precision@10 |
|---|---:|---:|
| TF-IDF | | |
| MiniLM | | |
| E5 | | |

Ten benchmark jest uproszczony: etykieta sentymentu pełni rolę przybliżonej oceny relewancji.


In [16]:
# Proszę przygotować zapytania, wyszukiwanie i funkcję precision_at_k(...).

# Zadanie 7: Wizualizacja reprezentacji

Dla reprezentacji E5 proszę zastosować PCA i zredukować wektory do dwóch wymiarów.

Proszę:

1. narysować wykres punktowy z kolorem odpowiadającym klasie;
2. podać udział wariancji wyjaśnionej przez dwie pierwsze składowe;
3. wskazać na wykresie lub opisać obszary, w których klasy wyraźnie się mieszają.

Nie należy traktować wykresu dwuwymiarowego jako pełnego obrazu oryginalnej przestrzeni.


In [17]:
# Proszę zaimplementować PCA i przygotować wykres.

# Zadanie 8: Analiza przypadków

Proszę przygotować tabelę zawierającą conajmniej 12 przypadków, które pokazują ograniczenia reprezentacji wektorowych.

Pierwsze 6 przypadków należy znaleźć w wynikach najbliższych sąsiadów lub wyszukiwania. Powinny to być przykłady, w których podobieństwo jest wysokie, ale wynik jest problematyczny z punktu widzenia sentymentu lub intencji zapytania.

Kolejne 6 przypadków należy przygotować jako pary kontrastowe, np.:

```text
Hotel jest dobry.
Hotel nie jest dobry.
```

Dla każdej pary proszę policzyć podobieństwo w MiniLM i E5.

Tabela powinna zawierać:

| nr | typ | tekst A | tekst B / wynik | MiniLM | E5 | Δ MiniLM | Δ E5 | komentarz |
|---:|---|---|---|---:|---:|---:|---:|---|

Cosinusy E5 są skupione wysoko, więc nie wolno porównywać bezwzględnych wartości MiniLM i E5. Δ to różnica między cosinusem pary a cosinusem pary niezwiązanej tematycznie (np. "Hotel jest dobry." i "Samochód jest szybki.") liczonym tym samym modelem.

W komentarzu proszę używać krótkich kategorii, np.:

- negacja;
- wspólny temat;
- wspólne słownictwo;
- podobna forma;
- różny sentyment;
- bardzo krótki tekst;
- inna.


In [18]:
# Proszę znaleźć 6 przypadków z danych, przygotować 6 par kontrastowych i zbudować tabelę 12 przypadków.

# Zadanie 9: Interpretacja wyników

To zadanie nie wymaga nowej implementacji.

Na podstawie wyników zadań 1-8 proszę odpowiedzieć:

1. W jakich sytuacjach TF-IDF zachowywał się lepiej od reprezentacji gęstych?
2. W jakich sytuacjach przewagę miały MiniLM lub E5?
3. Czy wysoka wartość `purity@k` oznacza, że reprezentacja jest ogólnie "lepsza semantycznie"?
4. Czy model o najlepszym `Precision@10` musi być najlepszym wyborem do klasyfikacji sentymentu?
5. Jakie dwa ograniczenia eksperymentu najbardziej wpływają na interpretację wyników?

Odpowiedzi powinny odwoływać się do uzyskanych liczb i przykładów.

# Zadanie 10: Decyzja projektowa

To zadanie nie wymaga nowej implementacji.

Proszę wybrać reprezentację dla każdego z trzech zastosowań:

1. szybkie wyszukiwanie po dokładnych nazwach i terminach;
2. wyszukiwanie semantyczne parafraz;
3. przyszły system RAG.

Dla każdego zastosowania proszę podać:

- wybraną reprezentację;
- jeden wynik ilościowy (jakość lub koszt) uzasadniający decyzję;
- jeden obserwowany przypadek błędu lub ograniczenie;
- koszt, który należy zaakceptować.

Na końcu proszę odpowiedzieć jednym akapitem na pytanie:

**Dlaczego embeddingu nie należy utożsamiać ze "znaczeniem tekstu"?**

# Karta modelu embeddingowego

Proszę wybrać jeden z dwóch modeli gęstej reprezentacji jako model rekomendowany do dalszych laboratoriów i utworzyć:

```text
artifacts/embedding_model_card.md
```

Karta powinna zawierać:

## Model
- nazwa checkpointu;
- wymiar reprezentacji;
- sposób normalizacji;
- wymagane prefiksy wejścia, jeśli dotyczy.

## Dane
- użyty zbiór;
- wielkość próbki;
- odsetek tekstów obciętych przez okno modelu;
- sposób doboru próbki.

## Koszt
- czas utworzenia reprezentacji;
- ms/dokument;
- rozmiar reprezentacji w pamięci.

## Jakość sąsiedztwa
- `purity@1`;
- `purity@5`;
- `purity@10`.

## Wyszukiwanie
- `Precision@5`;
- `Precision@10`.

## Analiza jakościowa
- najważniejsze obserwacje z PCA;
- dominujące typy błędów z analizy 20 przypadków.

## Zastosowanie
- do jakich zadań model jest rekomendowany;
- do jakich zadań nie należy go wybierać wyłącznie na podstawie tego laboratorium.

## Ograniczenia
- ograniczenia danych;
- ograniczenia benchmarku;
- ograniczenia samej reprezentacji.


# Wnioski

Na koniec pracy proszę zapisać krótkie wnioski. Każdy wniosek powinien wynikać z wykonanego eksperymentu i zawierać odwołanie do wyniku liczbowego, obserwacji albo konkretnego przypadku.

Nie należy przepisywać definicji TF-IDF, embeddingów ani podobieństwa cosinusowego.

# Warunki zaliczenia laboratorium

Laboratorium jest ukończone, jeśli zawiera:

- [ ] własną implementację eksperymentu TF-IDF i podobieństwa cosinusowego;
- [ ] reprezentacje MiniLM i E5;
- [ ] deterministyczną próbkę danych;
- [ ] benchmark czasu i pamięci trzech reprezentacji;
- [ ] implementację wyszukiwania najbliższych sąsiadów;
- [ ] `purity@1`, `purity@5`, `purity@10`;
- [ ] osiem zapytań w benchmarku wyszukiwania;
- [ ] własną implementację `Precision@k`;
- [ ] `Precision@5` i `Precision@10`;
- [ ] wizualizację PCA;
- [ ] tabelę 12 przypadków błędów i par kontrastowych;
- [ ] sprawdzenie okna modeli (`max_seq_length`) i odsetka obciętych tekstów;
- [ ] purity z punktem odniesienia (losowi sąsiedzi) i per klasa;
- [ ] odpowiedzi do zadania interpretacyjnego (Zadanie 9);
- [ ] plik `artifacts/embedding_model_card.md`;
- [ ] od pięciu do ośmiu wniosków końcowych.